# CatalogUpdate AI: Colab Runner & Smoke Test

This notebook runs **CatalogUpdate AI** on a free Google Colab T4 GPU (16GB).
It clones the repository, installs pinned requirements, runs a smoke test on a sample catalog image, executes the batch processing pipeline, and launches the Gradio web interface with a public share URL.

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

In [ ]:
# 2. Clone Repository & Install Dependencies
!git clone https://github.com/Zeeshaan-23/Gen-AI_Final.git
%cd Gen-AI_Final
!pip install -q -r requirements.txt

In [ ]:
# 3. Install Requirements
!pip install -q -r requirements.txt

In [ ]:
# 4. Prepare Test Image in data/input
import os
from PIL import Image, ImageDraw

os.makedirs("data/input", exist_ok=True)
os.makedirs("data/output", exist_ok=True)

sample_path = "data/input/sample_chair.png"
if not os.path.exists(sample_path):
    img = Image.new("RGB", (512, 512), (250, 250, 250))
    draw = ImageDraw.Draw(img)
    # Chair seat & backrest
    draw.rounded_rectangle([150, 100, 362, 320], radius=15, fill=(50, 50, 60))
    # Chair legs (target for masking)
    draw.rectangle([170, 320, 190, 460], fill=(190, 190, 190))
    draw.rectangle([322, 320, 342, 460], fill=(190, 190, 190))
    img.save(sample_path)
    print(f"Created synthetic catalog test image: {sample_path}")
else:
    print(f"Using existing test image: {sample_path}")

In [ ]:
# 5. Smoke Test: Load Engine & Edit Single Image
from PIL import Image
from src.engine import CatalogEditor

print("Initializing CatalogEditor on CUDA...")
editor = CatalogEditor(device="cuda")

input_img = Image.open("data/input/sample_chair.png").convert("RGB")
print(f"Input image loaded: size={input_img.size}")

print("Executing spec change: 'chair legs' -> 'walnut wood'...")
output = editor.edit(
    image=input_img,
    part_text="chair legs",
    new_spec_prompt="solid walnut wood chair legs, natural dark grain, studio lighting, furniture catalog photo, 8k",
    negative_prompt="metal, chrome, plastic, blurry, distorted, seams, low quality",
    threshold=0.4,
    dilate_px=12,
    steps=30,
    guidance=7.5,
    seed=42
)

output["mask"].save("data/output/smoke_test_mask.png")
output["result"].save("data/output/smoke_test_result.png")
print("Single image test completed!")

In [ ]:
# 6. Visual Inspection: Original vs. Mask vs. Inpainted Result
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(input_img)
axes[0].set_title("Original Product Image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(output["mask"], cmap="gray")
axes[1].set_title("Auto-Mask (CLIPSeg: 'chair legs')", fontsize=12)
axes[1].axis("off")

axes[2].imshow(output["result"])
axes[2].set_title("Inpainted Result (Walnut Wood)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 7. Run Batch Pipeline (Folder in -> Folder out -> Manifest)
from src.batch import run_batch, zip_outputs
import json

print("Starting batch pipeline across data/input/...")
manifest = run_batch(
    input_dir="data/input",
    output_dir="data/output",
    part_text="chair legs",
    spec_prompt="modern solid walnut wood chair legs, rich natural wood grain, catalog studio lighting, high resolution",
    negative_prompt="metal, chrome, plastic, blurry, seams, distorted",
    params={"steps": 30, "guidance": 7.5, "seed": 42, "device": "cuda", "editor": editor}
)

print(f"\nBatch complete! Logged {len(manifest)} items:")
print(json.dumps(manifest, indent=2))

In [ ]:
# 8. Batch Results Grid: Before vs. Mask vs. After
import os
import matplotlib.pyplot as plt
from PIL import Image

success_items = [m for m in manifest if "success" in m["status"]]
if success_items:
    n_display = min(len(success_items), 5)
    fig, axes = plt.subplots(n_display, 3, figsize=(15, 5 * n_display))
    if n_display == 1:
        axes = [axes]

    for idx, item in enumerate(success_items[:n_display]):
        in_img = Image.open(os.path.join("data/input", item["name"])).convert("RGB")
        mask_stem = os.path.splitext(item["name"])[0]
        m_img = Image.open(f"data/output/masks/{mask_stem}_mask.png")
        out_img = Image.open(item["output_path"]).convert("RGB")

        axes[idx][0].imshow(in_img)
        axes[idx][0].set_title(f"Original: {item['name']}")
        axes[idx][0].axis("off")

        axes[idx][1].imshow(m_img, cmap="gray")
        axes[idx][1].set_title(f"Mask ({item['mask_coverage_pct']}% coverage)")
        axes[idx][1].axis("off")

        axes[idx][2].imshow(out_img)
        axes[idx][2].set_title(f"Edited: {item['prompt'][:35]}...")
        axes[idx][2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No items with status 'success' to display.")

In [ ]:
# 9. Package & Download Output Archive
zip_archive = zip_outputs("data/output")
print(f"Output archive generated: {zip_archive}")
# To download to local browser in Colab:
# from google.colab import files
# files.download(zip_archive)

In [ ]:
# 10. Launch Interactive Gradio Web App with Public Share URL
!python app.py --share